# Against the Odds

For each puzzle: **predict** and write your prediction in a Markdown cell, **simulate**,
**calculate** exactly, then **explain** any gap between your prediction and the
result. Implement the functions for [The office party](../problems/against-the-odds/office-party.md),
[Waiting for heads](../problems/against-the-odds/waiting-for-heads.md), or
[The 100 prisoners](../problems/against-the-odds/100-prisoners.md)
first; this notebook holds the investigations.

In [ ]:
import math
import random
from itertools import islice, product

import matplotlib.pyplot as plt
import pandas as pd

from pyproblems.birthday import (
    estimate_shared_birthday_probability,
    shared_birthday_probability,
    smallest_group_with_probability,
)
from pyproblems.coin_patterns import (
    average_waiting_time,
    coin_flips,
    expected_waiting_time,
    flips_until,
)
from pyproblems.prisoners import (
    clever_strategy,
    count_successes,
    cycle_lengths,
    estimate_survival,
    fixed_strategy,
    random_strategy,
    shuffled_drawers,
)

# Seed the random numbers so that results can be repeated.
random.seed(2026)

## The office party

Prediction: how large must a group be for a shared birthday to be more likely
than not? Why do you think so?

The plotting code is provided; your job is to fill in the tables of results.
Run the cells in order.

In [ ]:
def plot_probability(results, trials):
    """Plot a table with `people`, `simulated` and `exact` columns."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(results["people"], results["exact"], label="exact")
    ax.plot(
        results["people"],
        results["simulated"],
        "o",
        markersize=4,
        label=f"simulated ({trials:,} trials)",
    )
    ax.axhline(0.5, color="grey", linestyle=":")
    ax.set_xlabel("People in the group")
    ax.set_ylabel("Chance of a shared birthday")
    ax.legend()
    fig.tight_layout()
    return ax

In [ ]:
sizes = list(range(2, 61, 2))
trials = 100

results = pd.DataFrame({"people": sizes})

# TODO: add a "simulated" column with the estimated chance of a shared birthday
# for each group size (use `trials`), and an "exact" column with the
# exact chance. Then run the next cell to plot them. The simulated points should
# hug the exact curve: how many trials does that take?

In [ ]:
ax = plot_probability(results, trials)
results

### Why so few?

Compare the number of pairs in the group, `math.comb(n, 2)`, with the number of
days. Then find the group size for even odds as the number of days grows.

In [ ]:
def plot_group_size(table):
    """Plot a table with `days` and `group_size` columns on log axes."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(
        table["days"],
        table["group_size"],
        "o-",
        label="smallest group for even odds",
    )
    ax.set_xlabel("Days in the year")
    ax.set_ylabel("Group size")
    ax.legend()
    fig.tight_layout()
    return ax

In [ ]:
days = [10, 100, 1_000, 10_000, 100_000]

table = pd.DataFrame({"days": days})

# TODO: add a "group_size" column: the smallest group with an even chance of a
# match for each number of days. Then add a "pairs" column with the number of
# pairs in that group (math.comb). Compare pairs with days: what do you notice?
# Run the next cell to plot group size against days.

In [ ]:
ax = plot_group_size(table)
table

### Uneven birthdays

Prediction: do uneven birthday popularities make a match more or less likely?

In [ ]:
# TODO: simulate groups whose birthdays are drawn with unequal weights using
# random.choices(..., weights=...), and compare with the uniform case.

## Waiting for heads

Prediction: the average number of flips to see HH, and to see HT.

The plotting code is provided; your job is to fill in the tables of results.
Run the cells in order.

In [ ]:
def plot_waiting_times(table):
    """Plot a table with `pattern`, `simulated` and `exact` columns as bars."""
    ax = table.plot.bar(x="pattern", y=["simulated", "exact"], figsize=(9, 4))
    ax.set_xlabel("Pattern")
    ax.set_ylabel("Average flips until the pattern appears")
    return ax

In [ ]:
trials = 20_000

# TODO: estimate the average waiting time for "HH" and "HT" with
# average_waiting_time and `trials`, and compare them with your predictions and
# with expected_waiting_time.

In [ ]:
patterns = [
    "".join(flips) for length in (3, 4) for flips in product("HT", repeat=length)
]
table = pd.DataFrame({"pattern": patterns})

# TODO: add a "simulated" column (average_waiting_time, with 5_000 trials is
# plenty) and an "exact" column (expected_waiting_time) for every pattern. Then
# run the next cell to plot them. Which patterns are quick and which are slow,
# and what do the quick ones have in common?

In [ ]:
table = table.sort_values("exact")
ax = plot_waiting_times(table)
table

### Penney's game

You choose a three-flip pattern, then your opponent chooses a different one. The
pattern that appears first wins. Fill in the table of results; the plotting code
is provided.

In [ ]:
def plot_penney(results):
    """Plot a table with `first`, `second` and `first_win_rate` columns."""
    grid = results.pivot(index="first", columns="second", values="first_win_rate")
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(grid, vmin=0, vmax=1, cmap="RdBu")
    ax.set_xticks(range(len(grid.columns)), grid.columns)
    ax.set_yticks(range(len(grid.index)), grid.index)
    ax.set_xlabel("Pattern chosen second")
    ax.set_ylabel("Pattern chosen first")
    for i, row in enumerate(grid.to_numpy()):
        for j, rate in enumerate(row):
            if rate == rate:
                ax.text(j, i, f"{rate:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(image, label="How often the pattern chosen first wins")
    fig.tight_layout()
    return ax

In [ ]:
patterns = ["".join(flips) for flips in product("HT", repeat=3)]
games = 2_000

# TODO: build `results`, a table with one row for each of the 56 ordered pairs
# of different patterns and the columns "first", "second" and "first_win_rate".
# For each game, flip a coin many times to make one shared list of flips,
# list(islice(coin_flips(), 200)), and see which pattern appears first with
# flips_until. One way: collect a dictionary for each pair in a list, then call
# pd.DataFrame(rows). Then run the next cell to plot it. Is there a best first
# choice?

In [ ]:
ax = plot_penney(results)

## The 100 prisoners

Prediction: the chance that all 100 prisoners find their number if each opens
50 random drawers.

The plotting code is provided; your job is to fill in the tables of results.
Run the cells in order.

In [ ]:
def plot_survival(survival):
    """Plot a table with `prisoners`, `random`, `fixed` and `clever` columns."""
    ax = survival.plot(
        x="prisoners",
        y=["random", "fixed", "clever"],
        marker="o",
        ylim=(0, 1),
        figsize=(6, 4),
    )
    ax.set_xlabel("Number of prisoners")
    ax.set_ylabel("Chance that everyone survives")
    return ax

In [ ]:
prisoners = [10, 20, 50, 100]
trials = 1_000

survival = pd.DataFrame({"prisoners": prisoners})

# TODO: add a "random" column, a "fixed" column and a "clever" column with the
# estimated chance that everyone survives with each strategy, when every
# prisoner may open half the drawers (limit = prisoners // 2). Then run the
# next cell to plot them.

In [ ]:
ax = plot_survival(survival)
survival

### Alone and together

Compare how many prisoners find their number under each strategy, on average,
and then in distribution.

In [ ]:
def plot_counts(counts, prisoners):
    """Plot a histogram of each column of a table of prisoners found per game."""
    axes = counts.hist(
        bins=range(prisoners + 2),
        align="left",
        layout=(1, 3),
        figsize=(12, 3.5),
        sharey=True,
    )
    for ax in axes.flat:
        ax.set_xlabel("Prisoners who find their number")
    axes.flat[0].set_ylabel("Games")
    return axes

In [ ]:
prisoners, limit, games = 30, 15, 3_000
counts = pd.DataFrame(index=range(games))  # one row per game

# TODO: add a "random", a "fixed" and a "clever" column: in each game, shuffle
# the drawers once and record how many prisoners find their number with each
# strategy (count_successes). Compare the averages, then run the next cell to
# plot the distribution of each column.

In [ ]:
axes = plot_counts(counts, prisoners)
counts.describe().round(2)

### Loops

Use cycle_lengths to explain who succeeds, then use it to estimate the survival
chance without simulating any prisoners. Fill in the data; the plotting code is
provided.

In [ ]:
def plot_longest_loop(longest, prisoners):
    """Histogram of a Series of longest-loop lengths, marking half the drawers."""
    ax = longest.plot.hist(bins=20, range=(0, prisoners), figsize=(7, 3.5))
    ax.axvline(
        prisoners // 2,
        color="grey",
        linestyle=":",
        label="drawers each prisoner may open",
    )
    ax.set_xlabel("Length of the longest loop")
    ax.set_ylabel("Arrangements")
    ax.legend()
    return ax

In [ ]:
prisoners, limit, games = 12, 6, 300
arrangements = [shuffled_drawers(prisoners) for _ in range(games)]
loops = pd.DataFrame(index=range(games))  # one row per game

# TODO: add a "successes" column (count_successes with clever_strategy) and an
# "on_short_loops" column (the number of drawers on loops of length at most
# `limit`, from cycle_lengths) for each arrangement. Do they always agree?

Everybody survives only if no loop is longer than the number of drawers each
prisoner may open. Look at the length of the *longest* loop in many random
arrangements of 100 drawers. Predict first: how is it spread, and where does half
the drawers fall among the values?

In [ ]:
prisoners = 100
trials = 4_000
longest = pd.Series(dtype=int)

# TODO: replace `longest` with a pandas Series of the length of the longest loop
# (use cycle_lengths) in each of `trials` random arrangements of `prisoners`
# drawers. Then run the next cell to plot it. What share of the
# arrangements have no loop longer than half the drawers?

In [ ]:
ax = plot_longest_loop(longest, prisoners)

Now put the game and the loops side by side. For each number of drawers that
every prisoner may open, play the whole game many times with `estimate_survival`.
Separately, find the share of random arrangements whose longest loop is no
longer than that number. Predict first: how will the two curves compare?

In [ ]:
def plot_survival_by_limit(comparison):
    """Plot a table with `limit`, `played` and `from_loops` columns."""
    ax = comparison.plot(
        x="limit",
        y=["played", "from_loops"],
        style=["o", "-"],
        ylim=(0, 1),
        figsize=(6, 4),
    )
    ax.set_xlabel("Drawers each prisoner may open")
    ax.set_ylabel("Chance that everyone survives")
    return ax

In [ ]:
prisoners = 40
trials = 500
loop_trials = 5_000
limits = list(range(4, prisoners + 1, 4))

comparison = pd.DataFrame({"limit": limits})

# TODO: add a "played" column: estimate_survival with clever_strategy for each
# limit. Then add a "from_loops" column: the share of `loop_trials` random
# arrangements whose longest loop (from cycle_lengths) is at most that limit. You
# can find the longest loops once and reuse them for every limit. Then run the
# next cell to plot both.

In [ ]:
ax = plot_survival_by_limit(comparison)
comparison

In [ ]:
loop_survival = pd.DataFrame({"prisoners": [100, 1_000]})
trials = 4_000

# TODO: add a "from_loops" column: the chance that everyone survives when each
# prisoner may open half the drawers, estimated from cycle_lengths alone over
# `trials` random arrangements (no prisoners needed). What does it approach as
# the number of prisoners grows?